In [1]:
from Node import Node
import json
import os
import pickle

#file_paths
g4_path = "2-Build_Graph/data/g4.pkl"
question_path = "InfoSeek/sampled_questions.jsonl"
knn_path = "Evaluation/data/knn.jsonl"
oven_path = "InfoSeek/oven_images_sampled"

with open(g4_path, "rb") as f:
    nodes = pickle.load(f)

questions = {}
with open(question_path, "r", encoding="utf-8") as f:
    for line in f:
        line = json.loads(line)
        questions[line["data_id"]] = (line["question"], line["image_id"])

knn = {}
with open(knn_path, "r", encoding="utf-8") as f:
    for line in f:
        line = json.loads(line)
        knn[line["qid"]] = line["knn"]

images = {}
for img in os.listdir(oven_path):
    img_id, ext = img.split(".")
    if ext.lower() not in {"jpg", "jpeg"}:
        raise ValueError(f"Not image: {img}")
    images[img_id] = f"{oven_path}/{img}"


In [2]:
from LLM.qwen3_vl_reranker import Qwen3VLReranker

model = Qwen3VLReranker(model_name_or_path="Qwen/Qwen3-VL-Reranker-2B")

c:\Users\HP\Desktop\Projects\MMNodeRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 625/625 [00:00<00:00, 3748.09it/s]


In [5]:
import random
random_qid = random.choice(list(questions.keys()))
question = questions[random_qid][0]
img_path = images[questions[random_qid][1]]
query = {"text": question, "image": img_path}
neighbors = knn[random_qid]
neighbors_content = []
V_met = False
for nei in neighbors:
    if "V" in nei:
        V_met = True
        nei_content = {"image": nodes[nei].content}
    else:
        continue
        nei_content = {"text": nodes[nei].content}
    neighbors_content.append(nei_content)
    if len(neighbors_content) == 2:
        break

inputs = {
    "instruction": "Retrieve images or text relevant to the user's query.",
    "query": query,
    "documents": neighbors_content,
}


print(inputs)
V_met

{'instruction': "Retrieve images or text relevant to the user's query.", 'query': {'text': 'What is the conservation status of this insect? (The status is assigned by the international union for conservation of nature. Choose one among Endangered,Least Concern,Critically Endangered,extinct species,extinct in the wild,Vulnerable,Near Threatened,Data Deficient)', 'image': 'InfoSeek/oven_images_sampled/oven_00491924.jpg'}, 'documents': [{'image': 'InfoSeek/wikipedia_images_sampled/Q2406579.jpg'}, {'image': 'InfoSeek/wikipedia_images_sampled/Q2206513.jpg'}]}


True

In [6]:
scores = model.process(inputs)
print(scores)

[0.490234375, 0.439453125]


In [7]:
neighbors_content

[{'image': 'InfoSeek/wikipedia_images_sampled/Q2406579.jpg'},
 {'image': 'InfoSeek/wikipedia_images_sampled/Q2206513.jpg'}]